In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

In [2]:
# ORNSTEIN-UHLENBECK PARAMETERS
# Mean growth rate (hour^-1)
mu = 1.0
# Doubling time at mean growth rate: T_d = ln(2) / mu
T_d = np.log(2) / mu
# Correlation timescale: "a couple of cell doublings" ≈ 2 * T_d
tau = 2 * T_d
# Mean reversion rate: θ = 1/τ
theta = 1 / tau
# For OU process: fluctuation_scale = σ / sqrt(2θ)
# Therefore: σ = fluctuation_scale * sqrt(2θ)
fluctuation_scale = 0.2  # per hour
sigma = fluctuation_scale * np.sqrt(2 * theta)
print(f"Ornstein-Uhlenbeck parameters:")
print(f"Doubling time at μ = {mu} hour⁻¹: {T_d:.3f} hours")
print(f"τ: {tau:.3f} hours")
print(f"θ: {theta:.3f} hour⁻¹")
print(f"σ: {sigma:.3f} hour⁻¹")
T_total = 4.5 #Total time (hours)
dt = T_total / 1000  # time step (hours)
t = np.arange(0, T_total, dt)
n_points = len(t)
print(f"Δt: = {dt:.4f} hours")
print(f"Total simulation time: {T_total} hours")
print(f"Number of time points: {n_points}") 

Ornstein-Uhlenbeck parameters:
Doubling time at μ = 1.0 hour⁻¹: 0.693 hours
τ: 1.386 hours
θ: 0.721 hour⁻¹
σ: 0.240 hour⁻¹
Δt: = 0.0045 hours
Total simulation time: 4.5 hours
Number of time points: 1001


In [3]:
rng_data = np.random.RandomState(42)
rng_mcmc = np.random.RandomState(1234567)

In [4]:
def simulate_g_np(n_points, mu, theta, sigma, dt, rng):
    g = np.zeros(n_points)
    g[0] = rng.normal(
        loc=mu,
        scale=sigma / np.sqrt(2 * theta)
    )
    alpha = np.exp(-theta * dt)
    beta  = sigma * np.sqrt((1 - np.exp(-2 * theta * dt)) / (2 * theta))
    
    for i in range(n_points - 1):
        mean = alpha * g[i] + (1 - alpha) * mu
        g[i + 1] = rng.normal(mean, beta)
        
    return g

In [5]:
def Likelihood(y, dt, mu, theta, sigma, sigma_n):
    y = np.asarray(y, dtype=float)
    N = len(y)
    if theta <= 0 or sigma <= 0 or sigma_n <= 0:
        return -np.inf
    if N < 1:
        return -np.inf
    alpha = np.exp(-theta * dt)
    beta2 = (sigma**2 / (2.0 * theta)) * (1.0 - np.exp(-2.0 * theta * dt))
    m_filt = mu
    P_filt = sigma**2 / (2.0 * theta)
    loglik = 0.0
    # i=0 predictive for y0: N(m0, P0 + sigma_n^2)
    v0 = y[0] - m_filt
    S0 = P_filt + sigma_n**2
    if S0 <= 0 or not np.isfinite(S0):
        return -np.inf
    loglik += -0.5 * (np.log(2.0*np.pi) + np.log(S0) + (v0*v0)/S0)
    # update at i=0
    K0 = P_filt / S0
    m_filt = m_filt + K0 * v0
    P_filt = (1.0 - K0) * P_filt
    for i in range(1, N):
        # Predict
        m_pred = alpha * m_filt + (1.0 - alpha) * mu
        P_pred = alpha**2 * P_filt + beta2
        # Innovation
        v = y[i] - m_pred
        S = P_pred + sigma_n**2
        if S <= 0 or not np.isfinite(S):
            return -np.inf
        loglik += -0.5 * (np.log(2.0*np.pi) + np.log(S) + (v*v)/S)
        # Update
        K = P_pred / S
        m_filt = m_pred + K * v
        P_filt = (1.0 - K) * P_pred
    return loglik
def log_halfnormal_sigma_from_eta(eta, scale):
    s = np.exp(eta)
    return -0.5*(s/scale)**2 + eta 
def logprior_phi(phi, bounds=None):
    if bounds is None:
        return 0.0
    mu, eta_theta, eta_sigma, eta_n = phi
    if not (bounds['mu'][0] <= mu <= bounds['mu'][1]):
        return -np.inf
    if not (bounds['eta_theta'][0] <= eta_theta <= bounds['eta_theta'][1]):
        return -np.inf
    if not (bounds['eta_sigma'][0] <= eta_sigma <= bounds['eta_sigma'][1]):
        return -np.inf
    lp = 0.0
    lp += log_halfnormal_sigma_from_eta(eta_n, scale=0.05)
    return lp
def logpost_phi(phi, y, dt, bounds=None):
    mu, eta_theta, eta_sigma, eta_n = phi
    lp = logprior_phi(phi, bounds=bounds)
    if not np.isfinite(lp):
        return -np.inf
    theta = np.exp(eta_theta)
    sigma = np.exp(eta_sigma)
    sigma_n = np.exp(eta_n)
    ll = Likelihood(y, dt, mu, theta, sigma, sigma_n)
    if not np.isfinite(ll):
        return -np.inf
    return ll + lp 

In [6]:
def mh_samples_single(y, dt, phi0=None, prop_scales=None, bounds=None, n_steps=30000, rng=None):
    if phi0 is None:
        phi0 = np.array([1.0, np.log(0.7), np.log(0.240), np.log(0.005)], float)
    else:
        phi0 = np.array(phi0, float)
    if prop_scales is None:
        prop_scales = np.array([0.02, 0.06, 0.06, 0.06], float)
    else:
        prop_scales = np.array(prop_scales, float)
    phi = phi0.copy()
    logp = logpost_phi(phi, y, dt, bounds=bounds) 
    chain = np.zeros((n_steps, 4))
    logp_chain = np.zeros(n_steps)
    accepts = 0
    for k in range(n_steps):
        print(f"Step {k+1}/{n_steps}")
        chain[k] = phi
        logp_chain[k] = logp
        phi_star = phi + rng.normal(0.0, prop_scales, size=4)
        logp_star = logpost_phi(phi_star, y, dt, bounds=bounds)
        if np.isfinite(logp_star):
            log_r = logp_star - logp
            if np.log(rng.uniform()) < log_r:
                phi, logp = phi_star, logp_star
                accepts += 1
    acc_rate = accepts / float(n_steps)
    return chain, logp_chain, acc_rate

In [7]:
def chain_phi_to_theta(chain):
    mu = chain[:, 0]
    theta = np.exp(chain[:, 1])
    sigma = np.exp(chain[:, 2])
    sigma_n = np.exp(chain[:, 3])
    return mu, theta, sigma, sigma_n
def plot_traces(chain, true_mu, true_theta, true_sigma, true_sigma_n):
    mu, theta, sigma, sigma_n = chain_phi_to_theta(chain)
    fig, axs = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    axs[0].plot(mu, lw=0.5)
    axs[0].axhline(true_mu, color="black", linestyle="--", linewidth=1)
    axs[0].set_ylabel("mu")
    axs[1].plot(theta, lw=0.5)
    axs[1].axhline(true_theta, color="black", linestyle="--", linewidth=1)
    axs[1].set_ylabel("theta")
    axs[2].plot(sigma, lw=0.5)
    axs[2].axhline(true_sigma, color="black", linestyle="--", linewidth=1)
    axs[2].set_ylabel("sigma")
    axs[3].plot(sigma_n, lw=0.5)
    axs[3].axhline(true_sigma_n, color="black", linestyle="--", linewidth=1)
    axs[3].set_ylabel("sigma_n")
    axs[3].set_xlabel("iteration")
    plt.tight_layout()
    plt.show() 

In [8]:
def KalmanSmoother(y, dt, mu, theta, sigma, sigma_n):
    y = np.asarray(y, dtype=float)
    N = len(y)
    if theta <= 0 or sigma <= 0 or sigma_n <= 0:
        return None, None
    if N < 1:
        return None, None
    alpha = np.exp(-theta * dt)
    beta2 = (sigma**2 / (2.0 * theta)) * (1.0 - np.exp(-2.0 * theta * dt))
    M_filt = np.zeros(N)      # m_t|t
    P_filt_arr = np.zeros(N)  # P_t|t
    M_pred = np.zeros(N)      # m_t|t-1
    P_pred_arr = np.zeros(N)  # P_t|t-1
    m_now = mu
    P_now = sigma**2 / (2.0 * theta)
    M_pred[0] = m_now
    P_pred_arr[0] = P_now
    v0 = y[0] - m_now
    S0 = P_now + sigma_n**2
    K0 = P_now / S0
    m_now = m_now + K0 * v0
    P_now = (1.0 - K0) * P_now
    M_filt[0] = m_now
    P_filt_arr[0] = P_now
    for i in range(1, N):
        m_p = alpha * m_now + (1.0 - alpha) * mu
        P_p = alpha**2 * P_now + beta2
        M_pred[i] = m_p
        P_pred_arr[i] = P_p
        v = y[i] - m_p
        S = P_p + sigma_n**2
        K = P_p / S
        m_now = m_p + K * v
        P_now = (1.0 - K) * P_p
        M_filt[i] = m_now
        P_filt_arr[i] = P_now
    M_smooth = np.zeros(N)
    P_smooth = np.zeros(N)
    M_smooth[N-1] = M_filt[N-1]
    P_smooth[N-1] = P_filt_arr[N-1]
    for i in range(N-2, -1, -1):
        P_pred_next = P_pred_arr[i+1]
        J = (P_filt_arr[i] * alpha) / P_pred_next
        M_smooth[i] = M_filt[i] + J * (M_smooth[i+1] - M_pred[i+1])
        P_smooth[i] = P_filt_arr[i] + (J**2) * (P_smooth[i+1] - P_pred_next)
        
    return M_smooth, P_smooth
def rts_smoother(chain, y, dt, n_samples=1000):
    N = len(y)
    burn_in = 5000
    valid_chain = chain[burn_in:]
    indices = np.random.choice(len(valid_chain), size=n_samples, replace=False)
    selected_samples = valid_chain[indices]
    cloud_trajectories = np.zeros((n_samples, N))
    for i, phi in enumerate(selected_samples):
        m_smooth, _ = KalmanSmoother(y, dt, mu=phi[0], theta=np.exp(phi[1]), sigma=np.exp(phi[2]), sigma_n=np.exp(phi[3]))
        cloud_trajectories[i, :] = m_smooth
    posterior_mean = np.mean(cloud_trajectories, axis=0)
    return posterior_mean

In [9]:
def generate_and_plot_growth_cloud(chain, y, dt, n_samples=100, true_g=None):
    N = len(y)
    t = np.arange(N) * dt
    burn_in = 5000
    valid_chain = chain[burn_in:]
    indices = np.random.choice(len(valid_chain), size=n_samples, replace=False)
    selected_samples = valid_chain[indices]
    cloud_trajectories = np.zeros((n_samples, N))
    print(f"Reconstructing {n_samples} trajectories...")
    for i, phi in enumerate(selected_samples):
        mu_s, eta_theta_s, eta_sigma_s, eta_n_s = phi
        theta_s = np.exp(eta_theta_s)
        sigma_s = np.exp(eta_sigma_s)
        sigma_n_s = np.exp(eta_n_s)
        m_smooth, P_smooth = KalmanSmoother(y, dt, mu_s, theta_s, sigma_s, sigma_n_s)
        cloud_trajectories[i, :] = m_smooth
    cloud_std = np.std(cloud_trajectories, axis=0)
    avg_width = np.mean(cloud_std)
    print(f"Average Tube Width (Std Dev): {avg_width:.6f}")
    posterior_mean = np.mean(cloud_trajectories, axis=0) 
    plt.figure(figsize=(12, 7))
    plt.plot(t, cloud_trajectories.T, color='gray', alpha=0.5, linewidth=1)
    plt.plot(t, posterior_mean, color='blue', linewidth=2, label='Posterior Mean Estimate (g)') 
    plt.plot(t, y, 'g.', markersize=3, alpha=0.3, label='Noisy Data (y)')
    if true_g is not None:
        plt.plot(t, true_g, 'r--', linewidth=2, label='Ground Truth (g)')
    plt.fill_between(t, m_smooth - 1.96*np.sqrt(P_smooth), m_smooth + 1.96*np.sqrt(P_smooth), color='blue', alpha=0.2, label='95% Confidence')
    plt.title(f"Reconstructed Growth Rates (P(g|D))\nGenerated from {n_samples} MCMC samples")
    plt.xlabel("Time (hours)")
    plt.ylabel("Growth Rate (hour⁻¹)")
    cloud_proxy = mlines.Line2D([], [], color='black', alpha=0.5, label='Posterior Uncertainty Cloud')
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles=[cloud_proxy] + handles, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    #Zooming
    zoom_t_start = 3.0
    zoom_t_end = 3.5
    idx_start = int(zoom_t_start / dt)
    idx_end = int(zoom_t_end / dt)
    y_window = y[idx_start:idx_end]
    y_min = np.min(y_window)
    y_max = np.max(y_window)
    y_buffer = (y_max - y_min) * 0.15
    plt.xlim(zoom_t_start, zoom_t_end)
    plt.ylim(y_min - y_buffer, y_max + y_buffer)
    plt.show()

In [10]:
def moving_average(data, window_size):
    window = np.ones(window_size) / window_size
    return np.convolve(data, window, mode='same')

In [11]:
def plot_all_methods(chain, y, dt, true_g, n_samples=1000):
    my_best_estimate = rts_smoother(chain, y, dt, true_g, n_samples=1000)
    plt.figure(figsize=(14, 8))
    plt.plot(t, y, 'g.', markersize=3, alpha=0.15, label='Noisy Data')
    plt.plot(t, moving_average(y, 5), color='orange', linewidth=2, alpha=0.8, label='Moving Avg (Window=5)')
    plt.plot(t, moving_average(y, 9), color='brown', linewidth=2, alpha=0.8, label='Moving Avg (Window=9)')
    plt.plot(t, my_best_estimate, color='blue', linewidth=2.5, label='Bayesian Kalman Smoother (Yours)')
    plt.plot(t, true_g, 'r--', linewidth=2, label='Ground Truth')
    plt.title("Method Comparison: Bayesian Smoother vs. Moving Averages")
    plt.xlabel("Time (hours)")
    plt.ylabel("Growth Rate")
    plt.legend()
    plt.grid(True, alpha=0.3)
    zoom_t_start = 3.0
    zoom_t_end = 3.5
    idx_start = int(zoom_t_start / dt)
    idx_end = int(zoom_t_end / dt)
    y_window = y[idx_start:idx_end]
    y_min = np.min(y_window)
    y_max = np.max(y_window)
    y_buffer = (y_max - y_min) * 0.15
    plt.xlim(zoom_t_start, zoom_t_end)
    plt.ylim(y_min - y_buffer, y_max + y_buffer)
    plt.show()

In [13]:
def mh_samples_joint(Y, dt, phi0=None, prop_scales=None, bounds=None, n_steps=30000, rng=None, save_filename=None):    
    if phi0 is None:
        phi0 = np.array([1.0, np.log(0.7), np.log(0.240), np.log(0.005)], float)
    else:
        phi0 = np.array(phi0, float)
    if prop_scales is None:
        prop_scales = np.array([0.02, 0.06, 0.06, 0.06], float)
    else:
        prop_scales = np.array(prop_scales, float)
    def compute_joint_logp(phi_eval):
        total_logp = 0.0
        for y in Y:
            val = logpost_phi(phi_eval, y, dt, bounds=bounds)
            if not np.isfinite(val):
                return -np.inf
            total_logp += val
        return total_logp
    phi = phi0.copy()
    logp = compute_joint_logp(phi)
    chain = np.zeros((n_steps, 4))
    logp_chain = np.zeros(n_steps)
    accepts = 0
    for k in range(n_steps):
        if (k + 1) % 5000 == 0:
            print(f" -> Step {k+1}/{n_steps}, Joint Log-Likelihood: {logp:.2f}")
        chain[k] = phi
        logp_chain[k] = logp
        phi_star = phi + rng.normal(0.0, prop_scales, size=4)
        logp_star = compute_joint_logp(phi_star)
        if np.isfinite(logp_star):
            log_r = logp_star - logp
            if np.log(rng.uniform()) < log_r:
                phi, logp = phi_star, logp_star
                accepts += 1
    acc_rate = accepts / float(n_steps)
    print(f"Finished! Acceptance rate: {acc_rate:.3f}")
    np.savez(save_filename, chain=chain, logp_chain=logp_chain, acc_rate=acc_rate)
    print(f"Success! Joint MCMC data safely saved to '{save_filename}'.")
    return chain, logp_chain, acc_rate

In [12]:
def plot_voilin_rmsd(Y, chain_single, chain_multi, G, dt, true_sigma_noise, save_filename="msd_results.csv"):
    N_traj = len(Y)        
    windows = [3, 5, 7, 15, 50, 75, 100,150, 1000]
    results_final = []
    for i in range(N_traj):
        print(f"Simulating trajectory {i+1}/{N_traj}...")
        msd_noisy = np.sqrt(np.mean((Y[i] - G[i])**2))
        results_final.append({'Method': 'Raw Data', 'MSD': msd_noisy})
        chain_i = chain_single[i]
        est_bayes_single = rts_smoother(chain_i, Y[i], dt, n_samples=1000)
        msd_bayes_single = np.sqrt(np.mean((est_bayes_single - G[i])**2))
        results_final.append({'Method': 'Bayesian', 'MSD': msd_bayes_single})
        est_bayes_multi = rts_smoother(chain_multi, Y[i], dt, n_samples=1000)
        msd_bayes_multi = np.sqrt(np.mean((est_bayes_multi - G[i])**2))
        results_final.append({'Method': 'Bayesian Multi', 'MSD': msd_bayes_multi})
        for w in windows:
            est_ma = moving_average(Y[i], w)
            msd_w = np.sqrt(np.mean((est_ma - G[i])**2))
            results_final.append({'Method': f'MA-{w}', 'MSD': msd_w})
    print("Simulation and MSD calculations complete. Ready for plotting.")
    df = pd.DataFrame(results_final)
    df.to_csv(save_filename, index=False)
    print(f"Success! Data securely saved to '{save_filename}'.")
    method_order = ['Raw Data']  + ['Bayesian'] + ['Bayesian Multi'] + [f'MA-{w}' for w in windows]
    plt.figure(figsize=(12, 7))
    sns.violinplot(
        x='Method', 
        y='MSD', 
        data=df, 
        order=method_order,
        inner="point",      
        palette="coolwarm",
        cut=0               
    )
    plt.axhline(
        y=true_sigma_noise, 
        color='black', 
        linestyle='--', 
        linewidth=2, 
        label=f'True Machine Noise (sigma_n = {true_sigma_noise})',
        zorder=5
    )
    plt.title(f"Performance Comparison: Mean Square Deviation over {N_traj} Simulations", fontsize=14, fontweight='bold')
    plt.ylabel("Mean Square Deviation (MSD) vs. Truth", fontsize=12)
    plt.xlabel("Estimation Method", fontsize=12)
    plt.yscale('log')
    plt.grid(True, which="major", ls="-", alpha=0.3)
    plt.grid(True, which="minor", ls=":", alpha=0.2)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()